[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-ctr.ipynb)

# Full Project: Click-Through Rate Prediction

*AIBits Academy · Machine Learning End To End · Full Project*

10,000 ad impressions, an SVM vs. Logistic Regression comparison, and an honest, un-inflated ~72% accuracy ceiling from just 4 behavioural features.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['click_through_rate.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the ad-impression data** (10,000 rows; the target is `Clicked on Ad`).

In [ ]:
df = pd.read_csv('click_through_rate.csv')
print(df.shape)
df['Clicked on Ad'].value_counts()

> **Business Problem**
>
> An ad platform wants to predict which users are likely to click a given ad, so higher-probability impressions can be prioritized — directly improving campaign ROI without needing more ad inventory.

> **Dataset**
>
> **10,000 rows, 10 columns** — site engagement (time on site, daily internet usage), demographics (age, area income, gender), and the binary target `Clicked on Ad` (perfectly balanced: 5,083 vs. 4,917). [Dataset source →](https://statso.io/click-through-rate-analysis-case-study/)

## Step 1 — SVM with an RBF Kernel

Four numeric behavioural/demographic features, scaled (mandatory for any distance- or margin-based method, per the SVM page), fed into an RBF-kernel SVM:

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

features = ['Daily Time Spent on Site','Age','Area Income','Daily Internet Usage']
X_train, X_test, y_train, y_test = train_test_split(df[features], df['Clicked on Ad'],
                                                       test_size=0.25, random_state=42, stratify=df['Clicked on Ad'])
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm.fit(X_train_s, y_train)
preds = svm.predict(X_test_s)
proba = svm.predict_proba(X_test_s)[:,1]
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}   ROC-AUC: {roc_auc_score(y_test, proba):.4f}")

## Step 2 — A Necessary Baseline Comparison

Per the Ethics in ML page's "compare against a real baseline, not a strawman" principle, the same scaled features were also fed into plain Logistic Regression:

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
lr_preds = lr.predict(X_test_s)
lr_proba = lr.predict_proba(X_test_s)[:,1]
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}   ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")

The RBF-kernel SVM (72.0% accuracy) edges out plain Logistic Regression (71.0%) on accuracy, but Logistic Regression actually has a marginally *higher* ROC-AUC (0.7744 vs. 0.7728) — the two models are, honestly, performing almost identically here. This is a genuinely useful negative-ish result: the extra complexity of a kernel method buys essentially nothing on this particular feature set, and a simpler, faster, more interpretable Logistic Regression would be the more defensible production choice.

## Visualizing "Almost Identical"

Full 0–1 scale, exact values labelled — the bars really are this close on both metrics.

> **⚠ Resist the Urge to Over-Engineer**
>
> With only 4 numeric features and near-identical SVM/Logistic Regression performance, the honest conclusion is that these four behavioural signals have a real but limited ceiling for this task — roughly 72% accuracy, 0.77 AUC. Chasing a fancier model on the same features is unlikely to move this number much; the higher-leverage next step would be adding genuinely new features (e.g., time-of-day from the Timestamp column, or ad-topic category), not swapping algorithms.

## Key Business Takeaways

- SVM and Logistic Regression perform almost identically here (72.0% vs. 71.0% accuracy, 0.7728 vs. 0.7744 AUC) — a reminder that a more sophisticated model isn't automatically a better one on a given feature set.
- The dataset's perfect class balance (5,083 vs. 4,917) means accuracy is a trustworthy metric here, unlike the imbalanced cases covered on the Handling Imbalanced Data page.
- ~72% accuracy from behavioural/demographic features alone is a genuine, honestly-reported ceiling, not a number chosen because it looked good — consistent with this course's stance against cherry-picked performance claims (Ethics in ML page).

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The click rate

Store in `click_rate` the fraction of impressions that were clicked. The classes are nearly balanced.

In [ ]:
click_rate = None   # TODO


In [ ]:
try:
    check("about 50.8%", abs(click_rate - df["Clicked on Ad"].mean()) < 1e-12 and 0.45 < click_rate < 0.55)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
click_rate = float(df["Clicked on Ad"].mean())

```

</details>

### Exercise 2 · Medium · Beat the majority-class baseline

Store the accuracy of always predicting the majority class of `y_test` in `majority_acc`, the SVM's test accuracy in `svm_acc` and `svm_beats` = whether the SVM is better.

In [ ]:
majority_acc = svm_acc = svm_beats = None   # TODO


In [ ]:
try:
    check("baseline near 0.5", 0.45 < majority_acc < 0.6)
    check("SVM is better", svm_beats is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
majority_acc = max(y_test.mean(), 1 - y_test.mean())
svm_acc = accuracy_score(y_test, preds)
svm_beats = bool(svm_acc > majority_acc)

```

</details>

### Exercise 3 · Stretch · Who to show the ad to

Marketing can only show the ad to people the logistic model is at least 70% sure will click. Store the number of such test users in `n_flagged` and the share of them who really clicked in `precision_flagged` (use `lr_proba` and `y_test`). Use `np.nan` for the precision if nobody qualifies.

In [ ]:
n_flagged = precision_flagged = None   # TODO


In [ ]:
try:
    mask = lr_proba >= 0.7
    check("count", n_flagged == int(mask.sum()))
    check("precision", np.isnan(precision_flagged) if mask.sum() == 0 else abs(precision_flagged - np.asarray(y_test)[mask].mean()) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
mask = lr_proba >= 0.7
n_flagged = int(mask.sum())
precision_flagged = float(np.asarray(y_test)[mask].mean()) if n_flagged else np.nan

```

Raising the confidence cut-off shrinks the audience but raises the hit rate - the same precision/coverage trade-off as everywhere in classification.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Click-Through Rate Prediction**.*